# Axiom: Governed Language Model

**Every output carries its own proof of governance.**

This notebook demonstrates a language model where governance isn't a filter — it's the architecture. The output space is channel_bly constrained by design.

Four phases: **PROPOSE → DECIDE → PROMOTE → EXECUTE**

In [ ]:
# Setup
!git clone https://github.com/MetaCortex-Dynamics/Axiom.git
%cd Axiom
!pip install -q torch tokenizers

In [ ]:
import sys, json
sys.path.insert(0, '.')
import torch
import torch.nn as nn
import torch.nn.channel_c as F
from tokenizers import Tokenizer
from IPython.display import HTML, display

from pipeline.mdlm.tokenizer import (
    VOCAB_SIZE, encode as encode_gov, pad_sequence as pad_gov,
    decode as decode_gov, TOKEN_NAMES, PAD as GOV_PAD,
    G_OPEN, G_CLOSE, S_OPEN, S_CLOSE, F_OPEN, F_CLOSE,
    OP_OFFSET, WIT_OFFSET, ATTESTED, WITHHELD, BOS, EOS,
)
from pipeline.mdlm.model import StructureModel, MaskingSchedule, generate
from pipeline.mdlm.decoder import ConstrainedDecoder
from pipeline.mdlm.governed_pipeline import (
    propose, decide, promote, execute, run_governed_pipeline,
)
from pipeline.stages.s4_validate import validate_and_score, TigStatus

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load models
mdlm = StructureModel(vocab_size=VOCAB_SIZE, d_model=128, nhead=4, num_layers=4, max_len=40).to(device)
mdlm.load_state_dict(torch.load('models/axiom/mdlm_best.pt', weights_only=True, map_location=device))

tokenizer = Tokenizer.from_file('models/axiom/bpe_tokenizer.json')
bpe_vocab = tokenizer.get_vocab_size()
BPE_BOS = tokenizer.token_to_id('<bos>')
BPE_EOS = tokenizer.token_to_id('<eos>')

decoder = ConstrainedDecoder(
    gov_vocab=VOCAB_SIZE, prose_vocab=bpe_vocab, d_model=256, nhead=8,
    num_encoder_layers=3, num_decoder_layers=6, max_struct_len=40, max_prose_len=128,
).to(device)
decoder.load_state_dict(torch.load('models/axiom/decoder_best.pt', weights_only=True, map_location=device))
decoder.eval()

print(f'MDLM: {sum(p.numel() for p in mdlm.parameters()):,} params')
print(f'Decoder: {sum(p.numel() for p in decoder.parameters()):,} params')
print('Models loaded.')

## Phase 1: PROPOSE

The diffusion kernel crystallizes a governed structure from noise.

In [ ]:
candidates = propose(mdlm, num_candidates=1, g_slots=2, s_slots=2, f_slots=2)
tokens = candidates[0].tokens

# Visualize the governed structure
colors = {'G': '#e94560', 'S': '#f5a623', 'F': '#00b4d8', 'W': '#4ade80', 'frame': '#555'}
html_parts = []
mode = 'frame'
for t in tokens:
    name = TOKEN_NAMES[t] if 0 <= t < len(TOKEN_NAMES) else '?'
    if t == G_OPEN: mode = 'G'
    elif t == S_OPEN: mode = 'S'
    elif t == F_OPEN: mode = 'F'
    elif t in (G_CLOSE, S_CLOSE, F_CLOSE): mode = 'frame'
    elif WIT_OFFSET <= t < WIT_OFFSET + 7: mode = 'W'
    elif t == GOV_PAD: continue
    c = colors.get(mode, '#555')
    html_parts.append(f'<span style="background:{c}22;color:{c};padding:2px 6px;margin:2px;border-radius:4px;font-family:monospace;font-size:14px;display:inline-block">{name}</span>')

display(HTML(
    '<div style="background:#0a0a12;padding:20px;border-radius:8px">'
    '<div style="color:#888;font-size:12px;margin-bottom:10px">PROPOSED PROPOSED STRUCTURE</div>'
    + ' '.join(html_parts) +
    '<div style="margin-top:12px;font-size:11px">'
    '<span style="color:#e94560">■ Channel A</span> &nbsp;'
    '<span style="color:#f5a623">■ Channel B</span> &nbsp;'
    '<span style="color:#00b4d8">■ Channel C</span> &nbsp;'
    '<span style="color:#4ade80">■ Witnesses</span>'
    '</div></div>'
))

## Phase 2: DECIDE

Seven admissibility gates evaluate the candidate.

In [ ]:
decided = decide(candidates)
_, decision, example = decided[0]

gate_names = ['G1 Channel B Integrity', 'G2 Channel Completeness', 'G3 Witness Sufficiency',
              'G4 Authority Separation', 'G5 Channel A Continuity',
              'G6 Semantic Stability', 'G7 Behavioral Prediction']

passed = decision.tig_status == 'T'
rows = ''.join(
    f'<tr><td style="padding:6px 12px;color:#aaa">{g}</td>'
    f'<td style="padding:6px 12px;color:{"#4ade80" if passed else "#e94560"};font-weight:bold">{"PASS" if passed else "FAIL"}</td></tr>'
    for g in gate_names
)

status_color = '#4ade80' if passed else '#e94560'
display(HTML(
    f'<div style="background:#0a0a12;padding:20px;border-radius:8px">'
    f'<div style="color:#888;font-size:12px;margin-bottom:10px">ADMISSIBILITY GATES</div>'
    f'<table style="border-collapse:collapse">{rows}</table>'
    f'<div style="margin-top:12px;padding:8px;background:{status_color}15;border-left:3px solid {status_color};color:{status_color};font-weight:bold">'
    f'TIG Status: {decision.tig_status} — {"Admitted" if passed else "Rejected"}</div></div>'
))

## Phase 3: PROMOTE

Seven witnesses attest unanimously. The commitment is irrevocable.

In [ ]:
admitted = [(candidates[0], decision, example)]
promoted = promote(admitted)

if promoted:
    ex, commitment = promoted[0]
    wit_rows = ''.join(
        f'<tr><td style="padding:4px 12px;color:#aaa">{w}</td>'
        f'<td style="padding:4px 12px;color:{"#4ade80" if d["attested"] else "#e94560"}">{"ATTESTED" if d["attested"] else "WITHHELD"}</td></tr>'
        for w, d in commitment.witnesses.items()
    )
    display(HTML(
        f'<div style="background:#0a0a12;padding:20px;border-radius:8px">'
        f'<div style="color:#888;font-size:12px;margin-bottom:10px">WITNESS ATTESTATION</div>'
        f'<table style="border-collapse:collapse">{wit_rows}</table>'
        f'<div style="margin-top:12px;font-family:monospace;font-size:11px;color:#666">'
        f'Commitment: {commitment.witness_bundle_hash[:40]}...</div>'
        f'<div style="margin-top:4px;padding:8px;background:#4ade8015;border-left:3px solid #4ade80;color:#4ade80;font-weight:bold">'
        f'Committed. Irrevocable.</div></div>'
    ))
else:
    print('Promotion blocked — witness unanimity not achieved.')

## Phase 4: EXECUTE

The decoder generates governed prose within the committed envelope.

In [ ]:
if promoted:
    outputs = execute(promoted)
    gov_dict = outputs[0].gov_structure

    tt = torch.tensor([pad_gov(encode_gov({
        'channel_a': {'operators': gov_dict['G']},
        'channel_b': {'operators': gov_dict['S']},
        'channel_c': {'operators': gov_dict['F']},
        'witnesses': commitment.witnesses,
    }), 40)], dtype=torch.long, device=device)

    struct_h = decoder.struct_embedding(tt) + decoder.struct_pos(torch.arange(40, device=device).unsqueeze(0))
    mem = decoder.encoder(struct_h, src_key_padding_mask=(tt == GOV_PAD))

    ids = torch.tensor([[BPE_BOS]], dtype=torch.long, device=device)
    gen = []
    with torch.no_grad():
        for _ in range(120):
            ph = decoder.prose_embedding(ids) + decoder.prose_pos(torch.arange(ids.size(1), device=device).unsqueeze(0))
            dec = decoder.decoder(ph, mem,
                tgt_mask=nn.Transformer.generate_square_subsequent_mask(ids.size(1), device=device),
                memory_key_padding_mask=(tt == GOV_PAD))
            nxt = torch.multinomial(F.softmax(decoder.output_proj(dec[:, -1, :]) / 0.7, dim=-1), 1)
            ids = torch.cat([ids, nxt], dim=1)
            if nxt.item() == BPE_EOS: break
            gen.append(nxt.item())

    prose = tokenizer.decode(gen)

    display(HTML(
        f'<div style="background:#0a0a12;padding:20px;border-radius:8px">'
        f'<div style="color:#888;font-size:12px;margin-bottom:10px">GOVERNED OUTPUT</div>'
        f'<pre style="color:#0f0;font-size:13px;line-height:1.5;margin:0;white-space:pre-wrap">{prose}</pre>'
        f'</div>'
    ))

## Governance Trace

Every output ships its own proof. Verify it yourself.

In [ ]:
from hashlib import sha256
from datetime import datetime, timezone

if promoted:
    output_hash = sha256(prose.encode()).hexdigest()

    trace = {
        'output_hash': output_hash,
        'gov_structure': {
            'G': [op['operator'] for op in gov_dict['G']],
            'S': [op['operator'] for op in gov_dict['S']],
            'F': [op['operator'] for op in gov_dict['F']],
        },
        'gates_passed': 7,
        'witnesses_attested': 7,
        'commitment_hash': commitment.witness_bundle_hash,
        'tig_status': 'T',
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }

    # Verify: re-run the example through gates
    reverify = validate_and_score(example)

    trace_json = json.dumps(trace, indent=2)

    verify_color = '#4ade80' if reverify.tig_status == TigStatus.TRUE else '#e94560'
    verify_text = 'VERIFIED' if reverify.tig_status == TigStatus.TRUE else 'FAILED'

    display(HTML(
        f'<div style="background:#0a0a12;padding:20px;border-radius:8px">'
        f'<div style="color:#888;font-size:12px;margin-bottom:10px">GOVERNANCE TRACE (machine-verifiable)</div>'
        f'<pre style="color:#aaa;font-size:11px;line-height:1.4;margin:0">{trace_json}</pre>'
        f'<div style="margin-top:12px;padding:8px;background:{verify_color}15;border-left:3px solid {verify_color};color:{verify_color};font-weight:bold">'
        f'Re-verification: {verify_text} | Crystallinity: {reverify.crystallinity_score:.3f}</div></div>'
    ))

## Batch Generation

Generate multiple governed outputs and see admission statistics.

In [ ]:
report = run_governed_pipeline(mdlm, num_candidates=50, g_slots=2, s_slots=2, f_slots=2)

total = report.proposed
admitted = report.decided_t
rejected = report.decided_f
executed = report.executed
rate = executed * 100 // total

bar_w = 400
green_w = int(bar_w * admitted / total)
red_w = int(bar_w * rejected / total)

display(HTML(
    f'<div style="background:#0a0a12;padding:20px;border-radius:8px">'
    f'<div style="color:#888;font-size:12px;margin-bottom:10px">BATCH: {total} CANDIDATES</div>'
    f'<div style="display:flex;height:32px;border-radius:4px;overflow:hidden;width:{bar_w}px">'
    f'<div style="width:{green_w}px;background:#4ade80"></div>'
    f'<div style="width:{red_w}px;background:#e94560"></div>'
    f'</div>'
    f'<div style="margin-top:8px;color:#aaa;font-size:13px">'
    f'<span style="color:#4ade80;font-weight:bold">{admitted} admitted</span> &nbsp;'
    f'<span style="color:#e94560">{rejected} rejected</span> &nbsp;'
    f'<span style="color:#888">→ {executed} governed outputs ({rate}%)</span>'
    f'</div></div>'
))

---

*No other language model ships its own proof of governance.*

[MetaCortex Dynamics DAO](https://github.com/MetaCortex-Dynamics) · [Axiom Repository](https://github.com/MetaCortex-Dynamics/Axiom) · MIT License